# Tamil Grammar Dataset Generator

Generate ~20,000 Tamil grammar correction training examples using **Qwen3-8B** in 4-bit quantization.

### Pipeline

```
Qwen3-8B (4-bit)
  → Generate Tamil grammar examples (20 categories × ~1000 each)
  → Validate (JSON, Tamil characters, length, structure)
  → Deduplicate
  → Shuffle + 95/5 train/eval split
  → Save as JSONL + ZIP + Download
```

### Instructions

1. **Runtime → Change runtime type → T4 GPU**
2. First run with `TOTAL_SAMPLES = 100` to test.
3. Inspect outputs carefully.
4. Change to `TOTAL_SAMPLES = 20_000` and run all cells.

The script supports **resume** — if Colab disconnects, rerun and it picks up from the saved file.

---

## CELL 1 — Enable GPU

In Colab: **Runtime → Change runtime type → T4 GPU**

Then run this cell to verify.

In [ ]:
!nvidia-smi

---

## CELL 2 — Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece tqdm

---

## CELL 3 — Check versions

In [ ]:
import torch
import transformers
import accelerate
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("WARNING: GPU not available!")

---

## CELL 4 — Configuration

In [ ]:
from pathlib import Path
import os
import json
import re
import random
import time
import gc

# ============================================================
# MODEL
# ============================================================
MODEL_NAME = "Qwen/Qwen3-8B"

# ============================================================
# DATASET SETTINGS
# ============================================================
# FIRST TEST:
TOTAL_SAMPLES = 100
# AFTER TESTING, CHANGE TO:
# TOTAL_SAMPLES = 20_000

# Number of examples requested from Qwen in one generation
BATCH_SIZE = 20

# Train/evaluation split
TRAIN_RATIO = 0.95

# Random seed
SEED = 42
random.seed(SEED)

# ============================================================
# OUTPUT DIRECTORY
# ============================================================
OUTPUT_DIR = Path("/content/tamil_grammar_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = OUTPUT_DIR / "raw_dataset.jsonl"
TRAIN_FILE = OUTPUT_DIR / "train.jsonl"
EVAL_FILE = OUTPUT_DIR / "eval.jsonl"
FINAL_FILE = OUTPUT_DIR / "final_dataset.jsonl"
ZIP_FILE = Path("/content/tamil_grammar_dataset.zip")

print("Output directory:", OUTPUT_DIR)
print("Total samples:", TOTAL_SAMPLES)

---

## CELL 5 — Load Qwen3-8B in 4-bit

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# 4-BIT CONFIGURATION
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.eval()
print("Model loaded successfully!")

---

## CELL 6 — Check model memory

In [ ]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"GPU allocated: {allocated:.2f} GB")
    print(f"GPU reserved: {reserved:.2f} GB")

---

## CELL 7 — Tamil grammar categories

20 categories covering diverse Tamil grammar error types.

In [ ]:
GRAMMAR_CATEGORIES = [
    "எழுத்துப் பிழை",
    "சொல் பிழை",
    "வினைச்சொல் பயன்பாட்டுப் பிழை",
    "காலப் பிழை",
    "ஒருமை பன்மை பிழை",
    "வேற்றுமை உருபுப் பிழை",
    "இடைச்சொல் பயன்பாட்டுப் பிழை",
    "பெயர்ச்சொல் பயன்பாட்டுப் பிழை",
    "வாக்கிய அமைப்புப் பிழை",
    "சொல் வரிசைப் பிழை",
    "சந்திப் பிழை",
    "உருபுப் பிழை",
    "பால் மற்றும் எண்ணிக்கைப் பிழை",
    "காலம் மற்றும் வினைச்சொல் பிழை",
    "எழுத்து மற்றும் உச்சரிப்புப் பிழை",
    "பேச்சுத்தமிழ் மற்றும் எழுத்துத்தமிழ் வேறுபாடு",
    "மரியாதை மொழிப் பிழை",
    "எதிர்மறை வாக்கியப் பிழை",
    "கேள்வி வாக்கிய அமைப்புப் பிழை",
    "கலப்பு இலக்கணப் பிழை"
]

print(f"Grammar categories: {len(GRAMMAR_CATEGORIES)}")
for i, cat in enumerate(GRAMMAR_CATEGORIES, 1):
    print(f"  {i:2d}. {cat}")

---

## CELL 8 — System prompt and generation prompt

In [ ]:
SYSTEM_PROMPT = """
நீங்கள் தமிழ் மொழியில் நிபுணத்துவம் பெற்ற
தமிழ் இலக்கண ஆசிரியர் மற்றும் dataset உருவாக்குநர்.

ஒரு தமிழ் Grammar Correction LLM-ஐ fine-tune செய்வதற்காக
உயர்தரமான supervised training examples உருவாக்க வேண்டும்.

ஒவ்வொரு example-லும்:

1. பிழையான தமிழ் வாக்கியம்
2. சரியான தமிழ் வாக்கியம்
3. பிழையின் வகை
4. பிழைக்கான தெளிவான தமிழ் விளக்கம்

இருக்க வேண்டும்.

முக்கியமான விதிகள்:

- இயல்பான நவீன தமிழ் பயன்படுத்தவும்.
- அர்த்தமற்ற வாக்கியங்களை உருவாக்க வேண்டாம்.
- ஒரே வாக்கியத்தை சிறிய மாற்றங்களுடன் மீண்டும் உருவாக்க வேண்டாம்.
- ஒவ்வொரு example-மும் வேறுபட்டதாக இருக்க வேண்டும்.
- intermediate-level grammar errors உருவாக்கவும்.
- மிக எளிய spelling mistakes மட்டும் அதிகமாக உருவாக்க வேண்டாம்.
- வாக்கியத்தின் அடிப்படை அர்த்தத்தை correction-ல் பாதுகாக்கவும்.
- correction உண்மையான தமிழ் இலக்கணத்தின் அடிப்படையில் இருக்க வேண்டும்.
- explanation தமிழில் இருக்க வேண்டும்.
- input மற்றும் correction இரண்டும் இயல்பான தமிழ் வாக்கியங்களாக இருக்க வேண்டும்.
- தமிழ் Unicode பயன்படுத்தவும்.
- ஆங்கில வார்த்தைகளை தேவையில்லாமல் பயன்படுத்த வேண்டாம்.
- பெயர்கள், இடங்கள், கல்வி, வேலை, குடும்பம், பயணம்,
  தொழில்நுட்பம், செய்தி, தினசரி வாழ்க்கை போன்ற பல்வேறு contexts பயன்படுத்தவும்.

JSON array மட்டும் output செய்யவும்.

Markdown வேண்டாம்.
Code fence வேண்டாம்.
கூடுதல் explanation வேண்டாம்.

"""


def build_prompt(category, count):
    return f"""
தமிழ் இலக்கண correction dataset-க்காக
{count} தனித்துவமான examples உருவாக்கவும்.

Grammar error category:

{category}

ஒவ்வொரு object-க்கும் கீழ்கண்ட fields இருக்க வேண்டும்:

{{
    "instruction": "இந்த தமிழ் வாக்கியத்தில் உள்ள இலக்கணப் பிழையை திருத்தவும்.",
    "input": "பிழையான தமிழ் வாக்கியம்",
    "correction": "சரியான தமிழ் வாக்கியம்",
    "error_type": "{category}",
    "explanation": "இந்த பிழை ஏன் ஏற்பட்டது மற்றும் correction ஏன் சரியானது என்பதை தமிழில் விளக்கவும்."
}}

கூடுதல் விதிகள்:

1. எல்லா {count} examples-மும் வேறுபட்டதாக இருக்க வேண்டும்.
2. ஒரே sentence pattern மீண்டும் வரக்கூடாது.
3. intermediate-level grammar பயன்படுத்தவும்.
4. correction இயல்பான தமிழ் ஆக இருக்க வேண்டும்.
5. input மற்றும் correction-ன் அடிப்படை அர்த்தம் ஒன்றாக இருக்க வேண்டும்.
6. explanation சரியான இலக்கண காரணத்தை கூற வேண்டும்.
7. input-ல் குறைந்தது 4 தமிழ் சொற்கள் இருக்க வேண்டும்.
8. மிகவும் artificial sentences உருவாக்க வேண்டாம்.
9. JSON array மட்டும் return செய்யவும்.

Output:

"""

print("System prompt and build_prompt defined.")

---

## CELL 9 — JSON extraction helper

In [ ]:
def extract_json_array(text):
    """Try to extract a JSON array from model output, handling common issues."""
    text = text.strip()

    # Remove markdown fences
    text = text.replace("```json", "")
    text = text.replace("```JSON", "")
    text = text.replace("```", "")
    text = text.strip()

    # Find first [ and last ]
    start = text.find("[")
    end = text.rfind("]")

    if start == -1 or end == -1:
        return None

    json_text = text[start : end + 1]

    try:
        return json.loads(json_text)
    except json.JSONDecodeError:
        # Try repairing trailing commas
        json_text = re.sub(r",\s*]", "]", json_text)
        try:
            return json.loads(json_text)
        except Exception:
            return None


print("extract_json_array defined.")

---

## CELL 10 — Generate one batch

In [ ]:
def generate_batch(category, count=20):
    """Generate a batch of Tamil grammar examples using Qwen3-8B."""
    prompt = build_prompt(category, count)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    # Qwen3 chat template — disable thinking mode for fast generation
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5000,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1] :]
    response = tokenizer.decode(
        generated_ids, skip_special_tokens=True
    )

    return extract_json_array(response)


print("generate_batch defined.")

---

## CELL 11 — Test the model

**Do this before starting 20K generation.**

In [ ]:
test_data = generate_batch("\u0b95\u0bbe\u0bb2\u0baa\u0bcd \u0baa\u0bbf\u0b9c\u0bc8", 3)

print(json.dumps(test_data, ensure_ascii=False, indent=2))

---

## CELL 12 — Tamil character validation

In [ ]:
def tamil_character_count(text):
    """Count Tamil Unicode characters in text."""
    if not isinstance(text, str):
        return 0
    return sum(
        1
        for char in text
        if "\u0b80" <= char <= "\u0bff"
    )


def contains_tamil(text):
    """Check if text contains at least 3 Tamil characters."""
    return tamil_character_count(text) >= 3


print("Tamil validation functions defined.")

---

## CELL 13 — Validate each example

In [ ]:
REQUIRED_FIELDS = [
    "instruction",
    "input",
    "correction",
    "error_type",
    "explanation",
]


def validate_example(item):
    """Validate a single dataset example."""
    # Must be dictionary
    if not isinstance(item, dict):
        return False

    # Required fields
    for field in REQUIRED_FIELDS:
        if field not in item:
            return False
        if not isinstance(item[field], str):
            return False
        if not item[field].strip():
            return False

    input_text = item["input"].strip()
    correction = item["correction"].strip()
    explanation = item["explanation"].strip()

    # Tamil validation
    if not contains_tamil(input_text):
        return False
    if not contains_tamil(correction):
        return False
    if not contains_tamil(explanation):
        return False

    # Minimum length
    if len(input_text.split()) < 4:
        return False
    if len(correction.split()) < 4:
        return False

    # Prevent identical input/correction
    if input_text == correction:
        return False

    # Prevent extremely long examples
    if len(input_text) > 500:
        return False
    if len(correction) > 500:
        return False
    if len(explanation) > 1000:
        return False

    return True


print("validate_example defined.")

---

## CELL 14 — Duplicate handling

In [ ]:
seen_pairs = set()


def make_key(item):
    input_text = item["input"].strip().lower()
    correction = item["correction"].strip().lower()
    return (input_text, correction)


def is_unique(item):
    key = make_key(item)
    if key in seen_pairs:
        return False
    seen_pairs.add(key)
    return True


print("Duplicate handling defined.")

---

## CELL 15 — Resume support

If Colab disconnects, this loads existing data so you don't lose progress.

In [ ]:
def load_existing_dataset():
    """Load existing raw dataset for resume support."""
    existing = []
    if not RAW_FILE.exists():
        return existing

    print("Existing dataset found.")
    with open(RAW_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
                existing.append(item)
                seen_pairs.add(make_key(item))
            except Exception:
                pass

    print("Existing examples:", len(existing))
    return existing


print("Resume support defined.")

---

## CELL 16 — Main generation loop

In [ ]:
def generate_dataset(total_samples):
    """Generate the full dataset with resume support."""
    dataset = load_existing_dataset()

    if len(dataset) >= total_samples:
        print("Dataset already contains enough examples.")
        return dataset[:total_samples]

    # Open file in append mode
    with open(RAW_FILE, "a", encoding="utf-8") as f:
        while len(dataset) < total_samples:
            # Rotate categories
            category = GRAMMAR_CATEGORIES[
                len(dataset) % len(GRAMMAR_CATEGORIES)
            ]

            remaining = total_samples - len(dataset)
            current_batch_size = min(BATCH_SIZE, remaining)

            print()
            print("=" * 60)
            print(f"Progress: {len(dataset)}/{total_samples}")
            print(f"Category: {category}")
            print(f"Requesting: {current_batch_size}")

            try:
                batch = generate_batch(category, current_batch_size)

                if not batch:
                    print("No valid JSON returned.")
                    time.sleep(2)
                    continue

                added = 0
                for item in batch:
                    if not validate_example(item):
                        continue
                    if not is_unique(item):
                        continue

                    # Metadata
                    item["id"] = len(dataset) + 1
                    item["language"] = "ta"
                    item["task"] = "tamil_grammar_correction"

                    # Save immediately
                    f.write(
                        json.dumps(item, ensure_ascii=False) + "\n"
                    )
                    f.flush()

                    dataset.append(item)
                    added += 1

                    if len(dataset) >= total_samples:
                        break

                print(f"Valid added: {added}")
                print(f"Total: {len(dataset)}")

            except RuntimeError as e:
                print("Runtime error:", e)
                print("Cleaning GPU memory...")
                gc.collect()
                torch.cuda.empty_cache()
                time.sleep(3)

            except Exception as e:
                print("Generation error:", e)
                time.sleep(3)

    return dataset


print("generate_dataset defined.")

---

## CELL 17 — Start with 100 examples

**Run this first to test the pipeline.**

In [ ]:
TOTAL_SAMPLES = 100

dataset = generate_dataset(TOTAL_SAMPLES)

print()
print("=" * 60)
print("TEST GENERATION FINISHED")
print("=" * 60)
print("Examples:", len(dataset))
print("File:", RAW_FILE)

---

## CELL 18 — Inspect random examples

**Do not skip this step.** Manually review before generating 20K.

In [ ]:
sample_count = min(10, len(dataset))
samples = random.sample(dataset, sample_count)

for item in samples:
    print()
    print("=" * 80)
    print("ID:", item["id"])
    print("CATEGORY:", item["error_type"])
    print("\nINCORRECT:")
    print(item["input"])
    print("\nCORRECT:")
    print(item["correction"])
    print("\nEXPLANATION:")
    print(item["explanation"])

---

## CELL 19 — Generate the actual 20K

Once the 100-example test looks good, run this.

In [ ]:
TOTAL_SAMPLES = 20_000

dataset = generate_dataset(TOTAL_SAMPLES)

print()
print("=" * 60)
print("20K GENERATION COMPLETE")
print("=" * 60)
print("Total examples:", len(dataset))
print("Raw file:", RAW_FILE)

---

## CELL 20 — Dataset statistics

In [ ]:
from collections import Counter

category_counts = Counter(item["error_type"] for item in dataset)

print("Total examples:", len(dataset))
print()
print("Category distribution:")
print("-" * 50)
for category, count in category_counts.most_common():
    print(f"{category}: {count}")

---

## CELL 21 — Final validation pass

In [ ]:
valid_data = []
invalid_count = 0

for item in dataset:
    if validate_example(item):
        valid_data.append(item)
    else:
        invalid_count += 1

print("Original:", len(dataset))
print("Valid:", len(valid_data))
print("Invalid:", invalid_count)

---

## CELL 22 — Final duplicate removal

In [ ]:
unique_data = []
final_seen = set()

for item in valid_data:
    key = (item["input"].strip(), item["correction"].strip())
    if key in final_seen:
        continue
    final_seen.add(key)
    unique_data.append(item)

print("Before duplicate removal:", len(valid_data))
print("After duplicate removal:", len(unique_data))

---

## CELL 23 — Shuffle dataset

In [ ]:
random.seed(42)
random.shuffle(unique_data)
print("Dataset shuffled.")

---

## CELL 24 — Reassign IDs

In [ ]:
for i, item in enumerate(unique_data, start=1):
    item["id"] = i

print("IDs reassigned.")

---

## CELL 25 — Create SFT format

Converts raw data into instruction/input/output format for QLoRA training.

In [ ]:
sft_data = []

for item in unique_data:
    output = (
        f"சரியான வாக்கியம்: "
        f"{item['correction']}\n\n"
        f"பிஜை வகை: "
        f"{item['error_type']}\n\n"
        f"விளக்கம்: "
        f"{item['explanation']}"
    )

    sft_item = {
        "instruction": item["instruction"],
        "input": item["input"],
        "output": output,
    }
    sft_data.append(sft_item)

print("SFT examples:", len(sft_data))

---

## CELL 26 — Train/eval split

In [ ]:
random.seed(42)
random.shuffle(sft_data)

split_index = int(len(sft_data) * TRAIN_RATIO)
train_data = sft_data[:split_index]
eval_data = sft_data[split_index:]

print("Total:", len(sft_data))
print("Train:", len(train_data))
print("Eval:", len(eval_data))

---

## CELL 27 — Save JSONL files

In [ ]:
def save_jsonl(filepath, data):
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


save_jsonl(TRAIN_FILE, train_data)
save_jsonl(EVAL_FILE, eval_data)
save_jsonl(FINAL_FILE, sft_data)

print("Saved:")
print(TRAIN_FILE)
print(EVAL_FILE)
print(FINAL_FILE)

---

## CELL 28 — Inspect final training data

In [ ]:
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    for i in range(5):
        item = json.loads(f.readline())
        print()
        print("=" * 80)
        print("INSTRUCTION:")
        print(item["instruction"])
        print("\nINPUT:")
        print(item["input"])
        print("\nOUTPUT:")
        print(item["output"])

---

## CELL 29 — Check JSONL integrity

In [ ]:
def check_jsonl(filepath):
    total = 0
    errors = 0
    with open(filepath, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            try:
                json.loads(line)
                total += 1
            except Exception as e:
                errors += 1
                print(f"Error at line {line_number}: {e}")
    return total, errors


train_total, train_errors = check_jsonl(TRAIN_FILE)
eval_total, eval_errors = check_jsonl(EVAL_FILE)

print("TRAIN")
print(f"Records: {train_total}")
print(f"Errors: {train_errors}")
print()
print("EVAL")
print(f"Records: {eval_total}")
print(f"Errors: {eval_errors}")

---

## CELL 30 — Zip the dataset

In [ ]:
import shutil

if ZIP_FILE.exists():
    ZIP_FILE.unlink()

shutil.make_archive(
    "/content/tamil_grammar_dataset", "zip", OUTPUT_DIR
)

print("ZIP created:", ZIP_FILE)

---

## CELL 31 — Download

In [ ]:
from google.colab import files

files.download("/content/tamil_grammar_dataset.zip")

---

## CELL 32 — Save to Google Drive (recommended)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
DRIVE_DIR = Path("/content/drive/MyDrive/tamil_grammar_dataset")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(TRAIN_FILE, DRIVE_DIR / "train.jsonl")
shutil.copy(EVAL_FILE, DRIVE_DIR / "eval.jsonl")
shutil.copy(RAW_FILE, DRIVE_DIR / "raw_dataset.jsonl")
shutil.copy(FINAL_FILE, DRIVE_DIR / "final_dataset.jsonl")

print("Dataset backed up to Google Drive.")
print(DRIVE_DIR)

---

## Next Steps

After generating the dataset:

1. **Manually review** 100-200 examples for quality.
2. **Add human-verified corrections** if possible.
3. **Run QLoRA fine-tuning** on a Tamil-capable model using `train.jsonl`.

### Pipeline Summary

```
Qwen3-8B (4-bit inference)
  → Tamil grammar generation
  → JSON validation
  → Duplicate removal
  → 20,000 data
  → 95% / 5% split
  → train.jsonl + eval.jsonl
  → QLoRA training
  → Tamil Grammar LLM
```